In [ ]:
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load
model = load_model('../model/bilstm_sentiment_v6.keras')

with open('../tokenizer/tokenizer_v6.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

max_len = 50  

label_map = {0: 'negative', 1: 'positive', 2: 'neutral'}


Text     : could not be more excited
Negative : 0.000
Positive : 1.000
Neutral  : 0.000
Result   : positive

Text     : not feeling great today
Negative : 0.992
Positive : 0.008
Neutral  : 0.000
Result   : negative

Text     : never been so relieved
Negative : 0.001
Positive : 0.999
Neutral  : 0.000
Result   : positive

Text     : not the happiest about this
Negative : 0.941
Positive : 0.028
Neutral  : 0.031
Result   : negative

Text     : cannot be more grateful
Negative : 0.001
Positive : 0.999
Neutral  : 0.000
Result   : positive



In [2]:
#more weight for the sentences after contrast words

def predict(text, threshold=0.73, margin=0.15):
    contrast_words = [' but ', ' however ', ' although ', ' though ', ' yet ', ' on the other hand ', ' then again ', ' that said ', ' even so ',]
    
    label_map = {0: 'negative', 1: 'positive', 2: 'neutral'}
    
    # Check if contrast word exists
    has_contrast = any(word in text for word in contrast_words)
    
    if has_contrast:
        # Split into before and after
        for word in contrast_words:
            if word in text:
                before = text.split(word)[0]
                after = text.split(word)[1]
                break
        
        # Predict both halves
        def get_pred(t):
            seq = tokenizer.texts_to_sequences([t])
            pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
            return model.predict(pad, verbose=0)[0]
        
        pred_before = get_pred(before)
        pred_after = get_pred(after)
        
        # Weight after "but" more (0.6) than before (0.4)
        pred = (0.4 * pred_before) + (0.6 * pred_after)
        
        print(f"Text       : {text}")
        print(f"Before but : {before.strip()} → {label_map[np.argmax(pred_before)]}")
        print(f"After but  : {after.strip()} → {label_map[np.argmax(pred_after)]}")
    
    else:
        seq = tokenizer.texts_to_sequences([text])
        pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
        pred = model.predict(pad, verbose=0)[0]
        print(f"Text       : {text}")
    
    top_class = np.argmax(pred)
    confidence = np.max(pred)
    pos_neg_diff = abs(pred[1] - pred[0])
    
    if confidence < threshold:
        result = 'neutral'
    elif pos_neg_diff < margin and top_class != 2:
        result = 'neutral'
    else:
        result = label_map[top_class]
    
    print(f"Negative   : {pred[0]:.3f}")
    print(f"Positive   : {pred[1]:.3f}")
    print(f"Neutral    : {pred[2]:.3f}")
    print(f"Confidence : {confidence:.3f}")
    print(f"Result     : {result}\n")